# 04 · Frozen evaluation set + Naive RAG baseline (+ Direct LLM reference)
**Owner:** A builds the eval set (Phase 1); B owns the RAG systems from Phase 2.

1. Build `data/processed/eval_sets.parquet` (FROZEN; every Task-2 system is scored on it)
2. Build the knowledge base from the TRAIN split only
3. Retrieval sanity check (intent-proxy Recall@1/@5, MRR@5)
4. Run Direct LLM (reference) and Naive RAG (baseline) → `outputs/runs/*.jsonl`

LLM setup (Colab → 🔑 Secrets): `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`, then set
`LLM_PROVIDER` and `LLM_MODEL` below. Start with `LIMIT = 20` to check cost before the full run.

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: clone the team repo once per session, install requirements, load API keys from Secrets
    REPO_URL = "https://github.com/sh1or1tama1/banking-support-agent.git"
    if not os.path.exists("/content/banking-support-agent"):
        subprocess.run(["git", "clone", REPO_URL, "/content/banking-support-agent"], check=True)
    os.chdir("/content/banking-support-agent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
# os.environ["LLM_PROVIDER"] = "openai"      # or "anthropic"
# os.environ["LLM_MODEL"] = "<cheapest current small model>"
# os.environ["PRICE_IN_PER_M"], os.environ["PRICE_OUT_PER_M"] = "<usd>", "<usd>"   # from the pricing page
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import pandas as pd
from IPython.display import display
from src import config as C
from src.data.load import load_banking77
from src.data.eval_sets import build_eval_sets, human_eval_subset
from src.rag.retriever import build_kb, DenseRetriever, BM25Retriever, retrieval_proxy_metrics
from src.rag.systems import DirectLLM, NaiveRAG
from src.llm.client import LLMClient
from src.eval.harness import run_system, summarize, escalation_confusion

LIMIT = int(os.getenv("LIMIT")) if os.getenv("LIMIT") else None   # e.g. 20 for a first, cheap test run
# Prefix for run files. Use it for provisional runs (e.g. a local model) so they never collide with
# the official outputs/runs/direct_llm.jsonl and naive_rag.jsonl (run_system resumes existing files).
RUN_PREFIX = os.getenv("RUN_PREFIX", "")

/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Frozen evaluation set

In [2]:
splits = pd.read_parquet(C.SPLITS_FILE)
if C.EVAL_SETS_FILE.exists():
    ev = pd.read_parquet(C.EVAL_SETS_FILE)          # already frozen -> reuse
else:
    ev = build_eval_sets(splits, load_banking77())
    ev.to_parquet(C.EVAL_SETS_FILE, index=False)
    human_eval_subset(ev).to_csv(C.PROCESSED / "human_eval_subset.csv", index=False)
display(ev.groupby(["eval_set", "expected_action"]).size().rename("queries").reset_index())
display(ev.groupby("eval_set").head(2)[["eval_set", "query", "gold_intent", "expected_action"]])

,eval_set,expected_action,queries
0,ambiguous,clarify,30
1,b77_real_inscope,answer,100
2,b77_unsupported,escalate,122
3,bitext_core,answer,104
4,bitext_hard,answer,50
5,out_of_domain,escalate,30


,eval_set,query,gold_intent,expected_action
0,bitext_core,I need to activate a Master Cardonline,activate_card,answer
1,bitext_core,"I need to activate a card on mobile, can I get...",activate_card,answer
104,bitext_hard,"I do not have a password, will you help me to ...",get_password,answer
105,bitext_hard,need to set up a password how do i do it,set_up_password,answer
154,b77_real_inscope,There is a fee on my account. Why?,check_fees,answer
155,b77_real_inscope,Why am I being charged a fee for transferring ...,check_fees,answer
254,b77_unsupported,Where is my new card? I have been waiting a week!,NaN,escalate
255,b77_unsupported,i want to track the card you sent,NaN,escalate
376,out_of_domain,What's the weather going to be like in Chicago...,NaN,escalate
377,out_of_domain,Can you recommend a good sushi restaurant near...,NaN,escalate


## 2. Knowledge base (train split only) and retrievers

In [3]:
kb = build_kb(splits)
print("KB documents:", len(kb), "| version:", kb.attrs.get("version"))
dense = DenseRetriever(kb)
bm25 = BM25Retriever(kb)
print("embedding backend:", dense.embedder.backend)

KB documents: 16804 | version: kb_890dd4c49255e4ca


W1122 22:17:51.231000 4760 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8329.89it/s]

Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Batches:   1%|          | 1/132 [00:01<02:17,  1.05s/it]

... (62 outputs left out)

Batches:  97%|█████████▋| 128/132 [00:10<00:00, 18.62it/s]

Batches:  98%|█████████▊| 130/132 [00:10<00:00, 18.76it/s]

Batches: 100%|██████████| 132/132 [00:10<00:00, 12.39it/s]

embedding backend: sbert:sentence-transformers/all-MiniLM-L6-v2


In [4]:
answerable = ev[ev["expected_action"] == "answer"]
rows = []
for name, r in (("dense", dense), ("bm25", bm25)):
    for s, sub in answerable.groupby("eval_set"):
        rows.append({"retriever": name, "eval_set": s, **retrieval_proxy_metrics(r, sub, k=5)})
retr = pd.DataFrame(rows).round(3); display(retr)
retr.to_csv(C.TABLES / "retrieval_proxy_metrics.csv", index=False)
# Read: does the top document have the right intent? Low Recall@1 on b77_real_inscope / bitext_hard
# is exactly where an agentic system (query rewrite, hybrid search, grading) should help.

,retriever,eval_set,recall@1,recall@5,mrr@5,n
0,dense,b77_real_inscope,0.710,0.80,0.747,100
1,dense,bitext_core,1.000,1.00,1.000,104
2,dense,bitext_hard,0.920,1.00,0.957,50
3,bm25,b77_real_inscope,0.300,0.40,0.336,100
4,bm25,bitext_core,0.913,1.00,0.954,104
5,bm25,bitext_hard,0.860,0.98,0.909,50


## 3. Run Direct LLM (reference) and Naive RAG (baseline)

In [5]:
llm = LLMClient()
print("provider:", llm.provider, "| model:", llm.model)
systems = [DirectLLM(llm), NaiveRAG(llm, dense, k=C.RAG_TOP_K)]
runs = {s.name: run_system(s, ev, tag=RUN_PREFIX + s.name, limit=LIMIT) for s in systems}

provider: openai | model: gpt-5.4-nano


direct_llm:   0%|          | 0/418 [00:00<?, ?it/s]

direct_llm:   0%|          | 1/418 [00:01<13:32,  1.95s/it]

direct_llm:   0%|          | 2/418 [00:02<09:17,  1.34s/it]

direct_llm:   1%|          | 3/418 [00:04<09:09,  1.32s/it]

direct_llm:   1%|          | 4/418 [00:05<08:21,  1.21s/it]

... (831 outputs left out)

naive_rag: 100%|█████████▉| 416/418 [09:53<00:02,  1.45s/it]

naive_rag: 100%|█████████▉| 417/418 [09:54<00:01,  1.35s/it]

naive_rag: 100%|██████████| 418/418 [09:56<00:00,  1.30s/it]

naive_rag: 100%|██████████| 418/418 [09:56<00:00,  1.43s/it]

In [6]:
for name, run in runs.items():
    print(f"\n=== {name} ===")
    display(summarize(run, ev))
    display(escalation_confusion(run, ev))


=== direct_llm ===


,n,behaviour_accuracy,answer_rate,escalate_rate,clarify_rate,invented_specific_rate,action_claim_rate,json_parse_ok,mean_llm_calls,mean_tokens,total_cost_usd,median_wall_s
eval_set,,,,,,,,,,,,
ambiguous,30,1.0000,0.0000,0.0000,1.0000,0.0,0.0,1.0,1.0,417.2333,0.0048,1.023
b77_real_inscope,100,0.1700,0.1700,0.0300,0.8000,0.0,0.0,1.0,1.0,450.3200,0.0195,1.141
b77_unsupported,122,0.0000,0.0902,0.0000,0.9098,0.0,0.0,1.0,1.0,436.0492,0.0217,1.077
bitext_core,104,0.0481,0.0481,0.0192,0.9327,0.0,0.0,1.0,1.0,429.5673,0.0176,1.057
bitext_hard,50,0.0400,0.0400,0.0200,0.9400,0.0,0.0,1.0,1.0,426.8000,0.0082,1.038
out_of_domain,30,0.7333,0.0000,0.7333,0.2667,0.0,0.0,1.0,1.0,440.8333,0.0056,1.196


,tp,fp,fn,tn,precision,recall,f1
0,22,6,130,248,0.7857,0.1447,0.2444



=== naive_rag ===


,n,behaviour_accuracy,answer_rate,escalate_rate,clarify_rate,invented_specific_rate,action_claim_rate,json_parse_ok,mean_llm_calls,mean_tokens,total_cost_usd,median_wall_s
eval_set,,,,,,,,,,,,
ambiguous,30,0.8333,0.1667,0.0000,0.8333,0.0000,0.0,1.0,1.0,1084.9333,0.0094,1.1565
b77_real_inscope,100,0.4700,0.4700,0.1000,0.4300,0.0100,0.0,1.0,1.0,1178.3500,0.0376,1.5220
b77_unsupported,122,0.1230,0.0656,0.1230,0.8115,0.0082,0.0,1.0,1.0,1101.4754,0.0386,1.1255
bitext_core,104,0.8846,0.8846,0.0096,0.1058,0.0000,0.0,1.0,1.0,1208.6346,0.0420,1.6210
bitext_hard,50,0.9600,0.9600,0.0000,0.0400,0.0000,0.0,1.0,1.0,1221.9800,0.0203,1.5870
out_of_domain,30,0.7667,0.1667,0.7667,0.0667,0.0333,0.0,1.0,1.0,1068.7000,0.0099,1.3315


,tp,fp,fn,tn,precision,recall,f1
0,38,11,114,243,0.7755,0.25,0.3781


In [7]:
# Look at a few outputs side by side
cmp = ev[["eval_id", "eval_set", "query"]].merge(
    runs["direct_llm"][["eval_id", "action", "answer"]].rename(columns={"action": "direct_action", "answer": "direct_answer"}), on="eval_id"
).merge(runs["naive_rag"][["eval_id", "action", "answer", "contexts"]], on="eval_id")
display(cmp.groupby("eval_set").head(1))

,eval_id,eval_set,query,direct_action,direct_answer,action,answer,contexts
0,ev_0000,bitext_core,I need to activate a Master Cardonline,clarify,Sure—I can help. Which ACME Bank card do you m...,answer,Sure—here are the steps to activate your Maste...,"[bx_00747, bx_00898, bx_00117]"
104,ev_0104,bitext_hard,"I do not have a password, will you help me to ...",clarify,Sure—I can help with the password reset steps....,answer,Yes—I can help you reset it. Please follow the...,"[bx_21603, bx_20910, bx_21288]"
154,ev_0154,b77_real_inscope,There is a fee on my account. Why?,clarify,"I can help you figure out what the fee is, but...",answer,Sorry about the confusion—account fees can var...,"[bx_11608, bx_11016, bx_11177]"
254,ev_0254,b77_unsupported,Where is my new card? I have been waiting a week!,clarify,"I can help you check the status, but I need on...",clarify,"I can help, but I need one detail: are you ask...","[bx_00937, bx_00333, bx_00456]"
376,ev_0376,out_of_domain,What's the weather going to be like in Chicago...,escalate,"I can’t help with weather forecasts. I can, ho...",escalate,"I can’t help with weather forecasts. I can, ho...","[bx_19930, bx_03724, bx_17277]"
406,ev_0406,ambiguous,something is wrong with my card,clarify,"I’m sorry you’re having trouble. To help, what...",clarify,I’m sorry you’re dealing with that. To help yo...,"[bx_05804, bx_04827, bx_04877]"
